# RF-DETR Multi-Class Detection Pipeline: Auto-Discovered Categories
### Model: RF-DETR Base (`resolution=560`, `optimizer=AdamW`, `lr=1e-4`, `grad_accumulation=8`)
This pipeline fine-tunes an RF-DETR Base model on multi-class annotations using:
- **Custom Training Architecture**: `load_coco_info`, `visualize_samples`, `_fixed_reinitialize` detection head patch, `COCODetectionDataset`, `build_criterion_and_postprocessors`, `BestValLossCheckpointSD`, AMP (`GradScaler`), and `MeanAveragePrecision(class_metrics=True)`.
- **Dynamic Category Auto-Discovery & Stratification**: Equal category proportions maintained across Train, Val, and Test splits.
- **Sample vs Full Mode**: Toggle `SAMPLE_SIZE = 1000` for fast pipeline testing vs `SAMPLE_SIZE = None` for complete dataset training.
- **Effective Batch Size**: `BATCH_SIZE = 4` with `GRAD_ACCUMULATION = 8` (effective batch size = 32).
- **VRAM Monitor**: Real-time tracking of GPU allocated, reserved, peak, and free memory.

In [ ]:
# STEP 0 - Install Dependencies (RF-DETR 1.4.0, Torchmetrics & CUDA Stack)
print("=" * 70)
print("[STEP 0] Verifying and installing dependencies...")
print("=" * 70)

!pip install -q --no-cache-dir "torch==2.5.1" "torchvision==0.20.1" --index-url https://download.pytorch.org/whl/cu121
!pip install -q --no-cache-dir "rfdetr==1.4.0"
!pip install -q --no-cache-dir "supervision>=0.22.0"
!pip install -q --no-cache-dir torchmetrics
!pip install -q --no-cache-dir pycocotools pandas numpy opencv-python Pillow matplotlib python-dotenv azure-storage-blob requests tqdm

print("Dependencies verified successfully.")

In [ ]:
# CELL 1 - Imports, VRAM Monitor & Logger Initialization
import os, sys, json, time, math, copy, logging, random, shutil
from pathlib import Path
from typing import Dict, List, Any, Optional, Tuple
from urllib.parse import urlparse
from concurrent.futures import ThreadPoolExecutor, as_completed

if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(line_buffering=True)
os.environ["PYTHONUNBUFFERED"] = "1"

# Configure multiprocessing sharing strategy to prevent 'Too many open files' error
import torch.multiprocessing as mp
try:
    mp.set_sharing_strategy('file_system')
except Exception:
    pass

try:
    import resource
    rlimit = resource.getrlimit(resource.RLIMIT_NOFILE)
    resource.setrlimit(resource.RLIMIT_NOFILE, (max(rlimit[0], 4096), max(rlimit[1], 4096)))
except Exception:
    pass

import cv2, torch, requests, supervision as sv
import matplotlib.pyplot as plt, matplotlib.patches as patches, numpy as np, pandas as pd
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.functional as TF
from PIL import Image
from dotenv import load_dotenv
from tqdm.auto import tqdm

try:
    from torchmetrics.detection.mean_ap import MeanAveragePrecision
    torchmetrics_status = "Available"
except ImportError:
    torchmetrics_status = "Not installed"

try:
    from rfdetr import RFDETRBase, RFDETRLarge
    from rfdetr import main as rfdetr_main
    from rfdetr.models.lwdetr import build_criterion_and_postprocessors
    rfdetr_status = "Available"
except ImportError:
    rfdetr_status = "Not installed"

# Auto-flushing console and file logger
class FlushHandler(logging.StreamHandler):
    def emit(self, record):
        super().emit(record)
        self.flush()

class FlushFileHandler(logging.FileHandler):
    def emit(self, record):
        super().emit(record)
        self.flush()

logger = logging.getLogger("rfdetr_multi_class")
logger.setLevel(logging.INFO)
logger.handlers.clear()
logger.propagate = False
log_format = "%(asctime)s | %(levelname)-8s | %(message)s"
console_handler = FlushHandler(sys.stdout)
console_handler.setFormatter(logging.Formatter(log_format))
logger.addHandler(console_handler)

def print_vram_usage(tag="Status"):
    if torch.cuda.is_available():
        alloc = torch.cuda.memory_allocated() / (1024**3)
        total = torch.cuda.get_device_properties(0).total_memory / (1024**3)
        logger.info(f"[VRAM - {tag}] {torch.cuda.get_device_name(0)}: {alloc:.2f} GB / {total:.2f} GB allocated")
    else:
        logger.info(f"[VRAM - {tag}] Running on CPU")

# Pretrained Weights Candidate Readability Check
PRETRAINED_WEIGHTS_CANDIDATES = [
    os.path.expanduser("~/rf-detr-base-coco.pth"),
    "/home/jupyter/rf-detr-base-coco.pth",
    "rf-detr-base-coco.pth"
]

def check_pretrained_weights_readable(candidates: List[str]) -> Tuple[Optional[str], str]:
    for cand in candidates:
        cand_path = os.path.expanduser(cand)
        if os.path.exists(cand_path) and os.access(cand_path, os.R_OK) and os.path.getsize(cand_path) > 1024*1024:
            try:
                ckpt = torch.load(cand_path, map_location="cpu", weights_only=False)
                count = len(ckpt.get("model", ckpt).keys()) if isinstance(ckpt, dict) else "?"
                del ckpt
                return cand_path, f"Readable ({os.path.getsize(cand_path)/1e6:.1f} MB, {count} tensors)"
            except Exception as e:
                return None, f"Corrupt ({cand_path}): {e}"
    return None, f"Not found among: {candidates}"

READABLE_WEIGHTS_PATH, WEIGHTS_STATUS = check_pretrained_weights_readable(PRETRAINED_WEIGHTS_CANDIDATES)

logger.info("=" * 65)
logger.info(f"[CELL 1] Core Libraries & Logger Ready (PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()})")
if READABLE_WEIGHTS_PATH:
    logger.info(f"   - Weights Check:  [OK] {READABLE_WEIGHTS_PATH} ({WEIGHTS_STATUS})")
else:
    logger.warning(f"   - Weights Check:  [NOT FOUND] {WEIGHTS_STATUS}")
print_vram_usage("Init")
logger.info("=" * 65)

In [ ]:
# CELL 2 - Configuration & Mode Setup
SAMPLE_SIZE = 1000  # Set to None for full dataset training
PIPELINE_NAME = "multi_class_train_rfdetr"
MODE_TAG = f"sample_{SAMPLE_SIZE}" if SAMPLE_SIZE else "full_data"

REPO_ROOT = Path(".")
PIPELINE_DIR = REPO_ROOT / PIPELINE_NAME
INPUT_JSON_DIR = REPO_ROOT / "coco_files"

DATASET_DIR = os.path.join(PIPELINE_DIR, f"dataset_{MODE_TAG}")
IMAGES_DIR = PIPELINE_DIR / "images"
RAW_IMAGES_DIR = IMAGES_DIR
TRAIN_DIR, VAL_DIR, TEST_DIR = [os.path.join(DATASET_DIR, s) for s in ["train", "val", "test"]]
TRAIN_ANN, VAL_ANN, TEST_ANN = [os.path.join(d, "_annotations.coco.json") for d in [TRAIN_DIR, VAL_DIR, TEST_DIR]]

MODEL_SIZE, PRETRAINED, RESOLUTION = "base", True, 560
# Discovered dynamically in Cell 5

# Training Parameters (Optimized for fast throughput with user constraints)
EPOCHS = 100
BATCH_SIZE = 2          # 2 images per batch
GRAD_ACCUMULATION = 8   # Effective batch = 16
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 2         # User constraint

OUTPUT_DIR = os.path.join(PIPELINE_DIR, f"runs/rf_detr_{MODE_TAG}")
CHECKPOINT_DIR = os.path.join(OUTPUT_DIR, "checkpoints")
LOG_DIR = os.path.join(OUTPUT_DIR, "logs")
INFERENCE_OUTPUT_DIR = os.path.join(PIPELINE_DIR, f"inference_{MODE_TAG}")
FINAL_MODEL_DIR = os.path.join(PIPELINE_DIR, "model")

PRETRAINED_WEIGHTS = READABLE_WEIGHTS_PATH if READABLE_WEIGHTS_PATH else "rf-detr-base-coco.pth"
AZURE_CONNECTION_STRING_ENV = "AZURE_STORAGE_CONNECTION_STRING"

IMAGE_FIELD, CATEGORY_FIELD, BBOX_FIELD = "image_id", "category_id", "bbox"
TRAIN_RATIO, VALID_RATIO, TEST_RATIO, RANDOM_SEED = 0.80, 0.10, 0.10, 42
DOWNLOAD_WORKERS, CONFIDENCE, NMS_THRESHOLD = 16, 0.50, 0.50

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True
    if hasattr(torch, "set_float32_matmul_precision"):
        torch.set_float32_matmul_precision("medium")  # Fast TensorFloat32 math

logger.info(f"Configured: Pipeline={PIPELINE_NAME} ({MODE_TAG}) | Batch={BATCH_SIZE} (Eff={BATCH_SIZE*GRAD_ACCUMULATION}) | Workers={NUM_WORKERS} | Res={RESOLUTION}x{RESOLUTION}")

In [ ]:
# CELL 3 - Dynamic Folder Creation & File Logging
for p in [DATASET_DIR, IMAGES_DIR, OUTPUT_DIR, CHECKPOINT_DIR, LOG_DIR, INFERENCE_OUTPUT_DIR, FINAL_MODEL_DIR]:
    os.makedirs(p, exist_ok=True)

log_file_path = os.path.join(LOG_DIR, "pipeline.log")
for h in [h for h in logger.handlers if isinstance(h, (logging.FileHandler, FlushFileHandler))]:
    logger.removeHandler(h); h.close()

file_handler = FlushFileHandler(log_file_path, mode="a", encoding="utf-8")
file_handler.setFormatter(logging.Formatter(log_format))
logger.addHandler(file_handler)

def tail_log(n: int = 25):
    """View latest log entries from inside Jupyter."""
    if os.path.exists(log_file_path):
        with open(log_file_path, "r", encoding="utf-8") as f:
            print("".join(f.readlines()[-n:]))

load_dotenv()
conn_str = os.getenv(AZURE_CONNECTION_STRING_ENV)
try:
    from azure.storage.blob import BlobServiceClient
    blob_service_client = BlobServiceClient.from_connection_string(conn_str) if conn_str else None
except Exception as e:
    blob_service_client = None

logger.info(f"Directories ready. Active file logger writing to: {log_file_path}")

In [ ]:
# CELL 4 - Read all annotation JSON files from coco_files/
logger.info("=" * 70)
logger.info(f"[CELL 4] Searching for annotation JSON files in: {INPUT_JSON_DIR}")
logger.info("=" * 70)

if not INPUT_JSON_DIR.exists():
    INPUT_JSON_DIR.mkdir(parents=True, exist_ok=True)

json_files = sorted(list(INPUT_JSON_DIR.glob("*.json")))
logger.info(f"   - Found {len(json_files)} JSON file(s) in {INPUT_JSON_DIR}")

raw_records = []
for jf in json_files:
    logger.info(f"   - Reading: {jf.name} ({jf.stat().st_size / 1024:.1f} KB)")
    with open(jf, "r", encoding="utf-8") as f:
        content = f.read().strip()
        if not content:
            continue
        try:
            parsed = json.loads(content)
            if isinstance(parsed, list):
                raw_records.extend(parsed)
            elif isinstance(parsed, dict):
                if "annotations" in parsed and isinstance(parsed["annotations"], list):
                    raw_records.extend(parsed["annotations"])
                else:
                    raw_records.append(parsed)
        except json.JSONDecodeError:
            f.seek(0)
            for line_idx, line in enumerate(f):
                line = line.strip()
                if line:
                    try:
                        parsed_line = json.loads(line)
                        if isinstance(parsed_line, list):
                            raw_records.extend(parsed_line)
                        else:
                            raw_records.append(parsed_line)
                    except Exception as err:
                        logger.warning(f"Error decoding line {line_idx+1} in {jf.name}: {err}")

logger.info(f"Total raw annotation records loaded: {len(raw_records)} from {len(json_files)} file(s).")

In [ ]:
# CELL 5 - Parse Annotations & Auto-Discover Multiple Categories
logger.info("=" * 70)
logger.info("[CELL 5] Parsing annotations and discovering categories dynamically...")
logger.info("=" * 70)

image_annotations: Dict[str, List[Dict[str, Any]]] = {}
category_counts: Dict[str, int] = {}

for item in raw_records:
    if not isinstance(item, dict):
        continue
    img_id = item.get(IMAGE_FIELD)
    if not img_id:
        continue
    
    raw_cat = item.get(CATEGORY_FIELD, ["default"])
    if isinstance(raw_cat, list):
        cat_name = str(raw_cat[0]) if len(raw_cat) > 0 else "default"
    else:
        cat_name = str(raw_cat)
    category_counts[cat_name] = category_counts.get(cat_name, 0) + 1

    raw_bbox = item.get(BBOX_FIELD, [])
    if not (isinstance(raw_bbox, list) and len(raw_bbox) == 4):
        continue
    
    try:
        x, y, w, h = [float(v) for v in raw_bbox]
        if w <= 0 or h <= 0:
            continue
    except (ValueError, TypeError):
        continue

    ann_dict = {
        "bbox": [x, y, w, h],
        "category_name": cat_name,
        "area": float(item.get("area", w * h)),
        "segmentation": item.get("segmentation", [])
    }
    
    if img_id not in image_annotations:
        image_annotations[img_id] = []
    image_annotations[img_id].append(ann_dict)

AUTO_CATEGORIES = sorted(list(category_counts.keys()))
cat_to_id = {cat: idx for idx, cat in enumerate(AUTO_CATEGORIES)}
id_to_cat = {idx: cat for cat, idx in cat_to_id.items()}

for img_id, anns in image_annotations.items():
    for ann in anns:
        ann["category_id"] = cat_to_id[ann["category_name"]]

NUM_CLASSES = len(AUTO_CATEGORIES)
total_boxes = sum(len(v) for v in image_annotations.values())

logger.info(f"   - Unique images found:         {len(image_annotations)}")
logger.info(f"   - Total valid bounding boxes:  {total_boxes}")
logger.info(f"   - Discovered Categories ({NUM_CLASSES}):")
for cat, count in category_counts.items():
    logger.info(f"       - ID {cat_to_id[cat]}: '{cat}' ({count} boxes, {count/total_boxes*100:.1f}%)")
logger.info(f"Discovered {NUM_CLASSES} categories: {AUTO_CATEGORIES}. Total boxes: {total_boxes}")

In [ ]:
# CELL 6 - Group Annotations & Apply Stratified Sampling Mode
logger.info("=" * 70)
logger.info(f"[CELL 6] Applying mode selection: {MODE_TAG.upper()}")
logger.info("=" * 70)

all_image_urls = sorted(list(image_annotations.keys()))
random.seed(RANDOM_SEED)

image_primary_cat: Dict[str, str] = {}
for u in all_image_urls:
    anns = image_annotations[u]
    if anns:
        cats = [a["category_name"] for a in anns]
        image_primary_cat[u] = max(set(cats), key=cats.count)
    else:
        image_primary_cat[u] = AUTO_CATEGORIES[0]

if SAMPLE_SIZE is not None and len(all_image_urls) > SAMPLE_SIZE:
    selected_image_urls = []
    cat_to_images: Dict[str, List[str]] = {cat: [] for cat in AUTO_CATEGORIES}
    for u in all_image_urls:
        cat_to_images[image_primary_cat[u]].append(u)
    
    total_imgs = len(all_image_urls)
    for cat in AUTO_CATEGORIES:
        imgs = cat_to_images[cat]
        target_count = max(1, int(round((len(imgs) / total_imgs) * SAMPLE_SIZE)))
        selected = random.sample(imgs, min(len(imgs), target_count))
        selected_image_urls.extend(selected)
    
    if len(selected_image_urls) > SAMPLE_SIZE:
        selected_image_urls = random.sample(selected_image_urls, SAMPLE_SIZE)
    logger.info(f"   Sample Mode Active: Stratified selection of {len(selected_image_urls)} images.")
else:
    selected_image_urls = all_image_urls
    logger.info(f"   Full Data Mode: Using all {len(selected_image_urls)} images.")

image_urls = sorted(list(set(selected_image_urls)))
filtered_annotations = {u: image_annotations[u] for u in image_urls}
total_selected_boxes = sum(len(v) for v in filtered_annotations.values())

logger.info(f"   - Total images in pipeline:         {len(image_urls)}")
logger.info(f"   - Total bounding boxes in pipeline: {total_selected_boxes}")

In [ ]:
# CELL 7 - Azure URL Helpers & Download Function
logger.info("=" * 70)
logger.info("[CELL 7] Defining Azure Blob download and caching functions...")
logger.info("=" * 70)

IMAGES_DIR = PIPELINE_DIR / "images"

def parse_azure_url(url: str) -> Tuple[Optional[str], Optional[str]]:
    parsed = urlparse(url)
    parts = parsed.path.strip("/").split("/", 1)
    if len(parts) == 2:
        return parts[0], parts[1]
    return None, None

def download_image(url: str, dest_dir: Path) -> Tuple[str, Optional[Path], Optional[str]]:
    filename = Path(urlparse(url).path).name
    dest_path = dest_dir / filename
    if dest_path.exists() and dest_path.stat().st_size > 0:
        return url, dest_path, None

    container_name, blob_name = parse_azure_url(url)
    if blob_service_client and container_name and blob_name:
        try:
            blob_client = blob_service_client.get_blob_client(container=container_name, blob=blob_name)
            with open(dest_path, "wb") as f:
                f.write(blob_client.download_blob().readall())
            return url, dest_path, None
        except Exception as e:
            pass

    try:
        resp = requests.get(url, timeout=30)
        if resp.status_code == 200:
            with open(dest_path, "wb") as f:
                f.write(resp.content)
            return url, dest_path, None
        else:
            return url, None, f"HTTP {resp.status_code}"
    except Exception as e:
        return url, None, str(e)

logger.info("Image download functions ready.")

In [ ]:
# CELL 8 - Parallel Azure Image Downloads
logger.info("=" * 70)
logger.info(f"[CELL 8] Downloading {len(image_urls)} images ({DOWNLOAD_WORKERS} threads)...")
logger.info("=" * 70)

download_results: Dict[str, Dict[str, Any]] = {}
with ThreadPoolExecutor(max_workers=DOWNLOAD_WORKERS) as executor:
    futures = {executor.submit(download_image, url, IMAGES_DIR): url for url in image_urls}
    done_count = 0
    for future in as_completed(futures):
        url, path, err = future.result()
        download_results[url] = {"local_path": path, "error": err}
        done_count += 1
        if done_count % 200 == 0 or done_count == len(image_urls):
            logger.info(f"   - Download Progress: {done_count}/{len(image_urls)} processed ({done_count/len(image_urls)*100:.1f}%)...")

success_downloads = [u for u, res in download_results.items() if res["error"] is None]
failed_downloads = [u for u, res in download_results.items() if res["error"] is not None]

logger.info("Download Summary:")
logger.info(f"   - Successfully ready: {len(success_downloads)} images")
logger.info(f"   - Failed:             {len(failed_downloads)} images")

In [ ]:
# CELL 9 - Category-Stratified Train/Val/Test Split (Equal Proportions)
logger.info("=" * 70)
logger.info("[CELL 9] Performing Category-Stratified Split (Train 80%, Val 10%, Test 10%)...")
logger.info("=" * 70)

valid_urls = [u for u in image_urls if download_results.get(u, {}).get("error") is None]
random.seed(RANDOM_SEED)

cat_to_valid_imgs: Dict[str, List[str]] = {cat: [] for cat in AUTO_CATEGORIES}
for u in valid_urls:
    p_cat = image_primary_cat.get(u, AUTO_CATEGORIES[0])
    cat_to_valid_imgs[p_cat].append(u)

train_urls, val_urls, test_urls = [], [], []
for cat, imgs in cat_to_valid_imgs.items():
    random.shuffle(imgs)
    n = len(imgs)
    n_tr = int(n * TRAIN_RATIO)
    n_va = int(n * VALID_RATIO)
    train_urls.extend(imgs[:n_tr])
    val_urls.extend(imgs[n_tr:n_tr + n_va])
    test_urls.extend(imgs[n_tr + n_va:])

def count_category_boxes(urls: List[str]) -> Dict[str, int]:
    counts = {cat: 0 for cat in AUTO_CATEGORIES}
    for u in urls:
        for a in filtered_annotations.get(u, []):
            c = a["category_name"]
            counts[c] = counts.get(c, 0) + 1
    return counts

train_cat_boxes = count_category_boxes(train_urls)
val_cat_boxes = count_category_boxes(val_urls)
test_cat_boxes = count_category_boxes(test_urls)
total_cat_boxes = {cat: train_cat_boxes[cat] + val_cat_boxes[cat] + test_cat_boxes[cat] for cat in AUTO_CATEGORIES}

proportions_rows = []
for cat in AUTO_CATEGORIES:
    tot = max(1, total_cat_boxes[cat])
    proportions_rows.append({
        "Category": cat,
        "Total Boxes": total_cat_boxes[cat],
        "Train Count": train_cat_boxes[cat],
        "Train %": f"{(train_cat_boxes[cat]/tot)*100:.1f}%",
        "Val Count": val_cat_boxes[cat],
        "Val %": f"{(val_cat_boxes[cat]/tot)*100:.1f}%",
        "Test Count": test_cat_boxes[cat],
        "Test %": f"{(test_cat_boxes[cat]/tot)*100:.1f}%",
    })

proportions_df = pd.DataFrame(proportions_rows)
logger.info("Category Proportions Across Splits:")
logger.info("\n" + proportions_df.to_string(index=False))
logger.info(f"Total Images: Train={len(train_urls)}, Val={len(val_urls)}, Test={len(test_urls)}")
logger.info(f"Stratified split verified across {len(AUTO_CATEGORIES)} categories.")

In [ ]:
# CELL 10 - Prepare Split Annotation Directories (Zero-Copy Architecture)
logger.info("=" * 70)
logger.info("[CELL 10] Preparing split annotation directories (Zero-Copy Architecture)...")
logger.info("=" * 70)

for split_name in [TRAIN_DIR, VAL_DIR, TEST_DIR]:
    os.makedirs(split_name, exist_ok=True)
    images_link = os.path.join(split_name, "images")
    if not os.path.exists(images_link):
        try:
            os.symlink(os.path.abspath(IMAGES_DIR), images_link)
        except OSError:
            pass

logger.info(f"   - Shared images directory:  {IMAGES_DIR} ({len(os.listdir(IMAGES_DIR))} files)")
logger.info(f"   - Train split annotations:  {TRAIN_DIR} ({len(train_urls)} images)")
logger.info(f"   - Val split annotations:    {VAL_DIR} ({len(val_urls)} images)")
logger.info(f"   - Test split annotations:   {TEST_DIR} ({len(test_urls)} images)")
logger.info("Split directories ready with zero redundant image copying.")

In [ ]:
# CELL 11 - Build Multi-Class COCO Annotations
logger.info("=" * 70)
logger.info(f"[CELL 11] Building COCO annotations for {NUM_CLASSES} discovered categories...")
logger.info("=" * 70)

categories_def = [{"id": idx, "name": cat, "supercategory": "none"} for cat, idx in cat_to_id.items()]

def build_coco_for_split(urls: List[str], split_dir: str, ann_path: str) -> Dict[str, Any]:
    images_list = []
    annotations_list = []
    ann_id = 1

    for img_id_idx, url in enumerate(urls, start=1):
        local_path = download_results[url]["local_path"]
        if not (local_path and local_path.exists()):
            continue
        try:
            with Image.open(local_path) as im:
                width, height = im.size
        except Exception:
            width, height = 1920, 1080

        filename = local_path.name
        images_list.append({
            "id": img_id_idx,
            "file_name": filename,
            "width": int(width),
            "height": int(height),
            "original_url": url
        })

        for ann in filtered_annotations.get(url, []):
            x, y, w, h = ann["bbox"]
            cid = ann["category_id"]
            annotations_list.append({
                "id": ann_id,
                "image_id": img_id_idx,
                "category_id": int(cid),
                "bbox": [round(x, 2), round(y, 2), round(w, 2), round(h, 2)],
                "area": round(w * h, 2),
                "iscrowd": 0,
                "segmentation": []
            })
            ann_id += 1

    coco_dict = {
        "info": {"description": f"RF-DETR Multi-Class Dataset ({NUM_CLASSES} classes)", "version": "1.0"},
        "licenses": [],
        "images": images_list,
        "annotations": annotations_list,
        "categories": categories_def
    }
    
    with open(ann_path, "w", encoding="utf-8") as f:
        json.dump(coco_dict, f, indent=2)
    logger.info(f"   - Saved: {ann_path} ({len(images_list)} images, {len(annotations_list)} boxes)")
    return coco_dict

build_coco_for_split(train_urls, TRAIN_DIR, TRAIN_ANN)
build_coco_for_split(val_urls, VAL_DIR, VAL_ANN)
build_coco_for_split(test_urls, TEST_DIR, TEST_ANN)

logger.info("Multi-Class COCO JSON files created.")

In [ ]:
# CELL 12 - Dataset Inspection (Summary Statistics Table)
def load_coco_info(annotation_path: str) -> dict:
    """Parse a COCO annotation file and return summary statistics."""
    with open(annotation_path) as f:
        data = json.load(f)
    categories = {c["id"]: c["name"] for c in data.get("categories", [])}
    num_images = len(data.get("images", []))
    num_anns = len(data.get("annotations", []))
    ann_per_cat: dict = {}
    for ann in data.get("annotations", []):
        cat_name = categories.get(ann["category_id"], "unknown")
        ann_per_cat[cat_name] = ann_per_cat.get(cat_name, 0) + 1
    return {
        "num_images": num_images,
        "num_anns": num_anns,
        "num_classes": len(categories),
        "categories": categories,
        "ann_per_cat": ann_per_cat,
        "raw": data,
    }

# Validate all annotation files exist
for split, path in [("train", TRAIN_ANN), ("val", VAL_ANN), ("test", TEST_ANN)]:
    assert os.path.exists(path), f"Missing annotation file for '{split}': {path}"
logger.info("All annotation files verified.")

train_info = load_coco_info(TRAIN_ANN)
val_info = load_coco_info(VAL_ANN)
test_info = load_coco_info(TEST_ANN)

# NUM_CLASSES is derived from the training annotation file
NUM_CLASSES = train_info["num_classes"]

logger.info("=" * 58)
logger.info(f"{'Split': <10} {'Images': >8} {'Annotations': >14} {'Classes': >9}")
logger.info("-" * 58)
for name, info in [("train", train_info), ("val", val_info), ("test", test_info)]:
    logger.info(f"{name: <10} {info['num_images']: >8} {info['num_anns']: >14} {info['num_classes']: >9}")
logger.info("=" * 58)
logger.info(f"Classes ({NUM_CLASSES} total): {train_info['categories']}")

In [ ]:
# CELL 13 - Visualize Samples Overlaid with Ground-Truth Bounding Boxes
def visualize_samples(annotation_path: str, image_dir: str, num_samples: int = 4, seed: int = 42):
    """Display random training samples overlaid with ground-truth bounding boxes."""
    rng = np.random.default_rng(seed)
    with open(annotation_path) as f:
        data = json.load(f)

    categories = {c["id"]: c["name"] for c in data.get("categories", [])}
    img_id_map = {im["id"]: im for im in data.get("images", [])}
    img_anns: dict = {}
    for ann in data.get("annotations", []):
        img_anns.setdefault(ann["image_id"], []).append(ann)

    valid_ids = [
        iid for iid, im in img_id_map.items()
        if os.path.exists(os.path.join(image_dir, im["file_name"]))
    ]
    if not valid_ids:
        logger.warning("No valid local images found to visualize.")
        return
    
    ids = rng.choice(valid_ids, size=min(num_samples, len(valid_ids)), replace=False)
    n = len(ids)
    fig, axes = plt.subplots(1, n, figsize=(5 * n, 5))
    if n == 1:
        axes = [axes]
    cmap = plt.cm.get_cmap("tab20", max(NUM_CLASSES, 1))

    for ax, img_id in zip(axes, ids):
        meta = img_id_map[img_id]
        img_path = os.path.join(image_dir, meta["file_name"])
        if not os.path.exists(img_path):
            ax.text(0.5, 0.5, "image \nnot found", ha="center", va="center", transform=ax.transAxes)
            ax.axis("off")
            continue
        img = Image.open(img_path).convert("RGB")
        ax.imshow(img)
        for ann in img_anns.get(img_id, []):
            x, y, w, h = ann["bbox"]
            cat_id = ann["category_id"]
            color = cmap(cat_id % max(NUM_CLASSES, 1))
            rect = patches.Rectangle((x, y), w, h, linewidth=2, edgecolor=color, facecolor="none")
            ax.add_patch(rect)
            ax.text(
                x, max(y - 4, 0), categories.get(cat_id, str(cat_id)),
                fontsize=8, color=color,
                bbox=dict(boxstyle="round,pad=0.1", fc="white", alpha=0.65, ec="none")
            )
        ax.set_title(meta["file_name"][:20], fontsize=8)
        ax.axis("off")

    plt.suptitle("Training Set Samples - Ground Truth Boxes", fontsize=12, y=1.01)
    plt.tight_layout()
    save_path = os.path.join(LOG_DIR, "sample_images.png")
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    logger.info(f"Visual sample images saved -> {save_path}")

visualize_samples(TRAIN_ANN, str(IMAGES_DIR))

In [ ]:
# CELL 14 - Safe Pretrained Weights Loading & Detection Head Patch
from rfdetr.models.lwdetr import LWDETR

# 1. Patch LWDETR.load_state_dict using torch.nn.Module.load_state_dict directly (prevents RecursionError)
def _safe_lwdetr_load_state_dict(self, state_dict, strict=True):
    """Filters out classification heads (class_embed, enc_out_class_embed) from checkpoint when num_classes differs."""
    model_state = self.state_dict()
    filtered_state_dict = {}
    mismatched = []
    
    for k, v in state_dict.items():
        clean_k = k
        if clean_k not in model_state and clean_k.startswith("model."):
            clean_k = clean_k[6:]
        if clean_k not in model_state and clean_k.startswith("module."):
            clean_k = clean_k[7:]
            
        if clean_k in model_state:
            if model_state[clean_k].shape == v.shape:
                filtered_state_dict[clean_k] = v
            else:
                mismatched.append((clean_k, tuple(v.shape), tuple(model_state[clean_k].shape)))
        elif k in model_state:
            if model_state[k].shape == v.shape:
                filtered_state_dict[k] = v
            else:
                mismatched.append((k, tuple(v.shape), tuple(model_state[k].shape)))

    if mismatched:
        logger.info(f"Notice: Filtered {len(mismatched)} classification head layers with class-count mismatch from COCO weights:")
        for name, ckpt_shape, model_shape in mismatched[:4]:
            logger.info(f"   - {name}: checkpoint {ckpt_shape} -> target model {model_shape}")
        if len(mismatched) > 4:
            logger.info(f"   - ... and {len(mismatched) - 4} more classification head tensors.")
        logger.info("Pretrained backbone, transformer, and regression heads loaded successfully.")
    
    # Call base PyTorch nn.Module.load_state_dict directly to avoid any recursion on re-runs
    return torch.nn.Module.load_state_dict(self, filtered_state_dict, strict=False)

LWDETR.load_state_dict = _safe_lwdetr_load_state_dict
logger.info("LWDETR.load_state_dict patched for safe custom num_classes loading.")

# 2. Patch reinitialize_detection_head to handle both class_embed and transformer.enc_out_class_embed
def _fixed_reinitialize(self, num_classes):
    lwdetr = self.model  # the actual LWDETR nn.Module
    device = next(lwdetr.parameters()).device
    
    # Reinitialize class_embed
    if hasattr(lwdetr, "class_embed"):
        in_feat = lwdetr.class_embed.in_features
        lwdetr.class_embed = nn.Linear(in_feat, num_classes).to(device)
        nn.init.normal_(lwdetr.class_embed.weight, std=0.01)
        nn.init.zeros_(lwdetr.class_embed.bias)
        
    # Reinitialize transformer.enc_out_class_embed
    if hasattr(lwdetr, "transformer") and hasattr(lwdetr.transformer, "enc_out_class_embed"):
        enc_heads = lwdetr.transformer.enc_out_class_embed
        if isinstance(enc_heads, nn.ModuleList):
            for i in range(len(enc_heads)):
                in_feat = enc_heads[i].in_features
                enc_heads[i] = nn.Linear(in_feat, num_classes).to(device)
                nn.init.normal_(enc_heads[i].weight, std=0.01)
                nn.init.zeros_(enc_heads[i].bias)
        elif isinstance(enc_heads, nn.Linear):
            in_feat = enc_heads.in_features
            lwdetr.transformer.enc_out_class_embed = nn.Linear(in_feat, num_classes).to(device)
            nn.init.normal_(lwdetr.transformer.enc_out_class_embed.weight, std=0.01)
            nn.init.zeros_(lwdetr.transformer.enc_out_class_embed.bias)
            
    logger.info(f"Detection heads (class_embed + enc_out_class_embed) initialized for num_classes={num_classes}")

rfdetr_main.Model.reinitialize_detection_head = _fixed_reinitialize
logger.info("reinitialize_detection_head patched.")

# 3. Model Instantiation
ModelClass = RFDETRLarge if MODEL_SIZE == "large" else RFDETRBase
model = ModelClass(
    num_classes=NUM_CLASSES,
    pretrain_weights=PRETRAINED_WEIGHTS,
    resolution=RESOLUTION
)
logger.info(f"RF-DETR-{MODEL_SIZE.capitalize()} instantiated with resolution={RESOLUTION}, num_classes={NUM_CLASSES}")

# Explicitly ensure detection heads match NUM_CLASSES
if hasattr(model, "model") and hasattr(model.model, "reinitialize_detection_head"):
    model.model.reinitialize_detection_head(NUM_CLASSES)

# Build ordered list of class names from COCO categories
class_names = [
    train_info["categories"][cid]
    for cid in sorted(train_info["categories"].keys())
]
logger.info(f"class_names: {class_names}")

In [ ]:
# CELL 15 - Optimized PyTorch Dataset (Fast OpenCV C++ Decoding)
class COCODetectionDataset(Dataset):
    """Fast COCO dataset using OpenCV for multi-threaded decoding and bilinear resizing."""
    def __init__(self, img_dir: str, ann_file: str, resolution: int = 560):
        with open(ann_file) as f:
            data = json.load(f)
        self.img_dir = img_dir
        self.resolution = resolution
        self.cat_to_id = {c: i for i, c in enumerate(sorted(c["id"] for c in data["categories"]))}
        ann_by_img: dict = {}
        for ann in data["annotations"]:
            ann_by_img.setdefault(ann["image_id"], []).append(ann)
        self.samples = [(img, ann_by_img[img["id"]]) for img in data["images"] if img["id"] in ann_by_img]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_meta, anns = self.samples[idx]
        img_path = os.path.join(self.img_dir, img_meta["file_name"])

        # High-performance OpenCV decode & resize (3x faster than PIL on CPU)
        cv_img = cv2.imread(img_path)
        if cv_img is not None:
            orig_h, orig_w = cv_img.shape[:2]
            resized = cv2.resize(cv_img, (self.resolution, self.resolution), interpolation=cv2.INTER_LINEAR)
            rgb = cv2.cvtColor(resized, cv2.COLOR_BGR2RGB)
            img_t = torch.from_numpy(rgb).permute(2, 0, 1).float().div_(255.0)
        else:
            with Image.open(img_path).convert("RGB") as pil_im:
                orig_w, orig_h = pil_im.size
                resized = pil_im.resize((self.resolution, self.resolution), Image.BILINEAR)
                img_t = TF.to_tensor(resized)

        img_t = TF.normalize(img_t, mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

        boxes, labels = [], []
        for ann in anns:
            x, y, w, h = ann["bbox"]
            if w <= 0 or h <= 0: continue
            boxes.append([
                float(np.clip((x + w / 2) / orig_w, 0, 1)),
                float(np.clip((y + h / 2) / orig_h, 0, 1)),
                float(np.clip(w / orig_w, 0, 1)),
                float(np.clip(h / orig_h, 0, 1))
            ])
            labels.append(self.cat_to_id[ann["category_id"]])

        return img_t, {
            "boxes": torch.tensor(boxes, dtype=torch.float32) if boxes else torch.zeros((0, 4)),
            "labels": torch.tensor(labels, dtype=torch.long) if labels else torch.zeros(0, dtype=torch.long),
            "image_id": torch.tensor([img_meta["id"]]),
            "orig_size": torch.tensor([orig_h, orig_w]),
            "size": torch.tensor([self.resolution, self.resolution]),
        }

def collate_fn(batch):
    return torch.stack([item[0] for item in batch]), [item[1] for item in batch]

logger.info("Fast COCODetectionDataset & collate_fn ready.")

In [ ]:
# CELL 16 - Criterion, Asynchronous DataLoaders, AdamW & Checkpoint
args = copy.deepcopy(model.model.args)
args.num_classes = NUM_CLASSES
args.device = "cuda" if torch.cuda.is_available() else "cpu"
criterion, _ = build_criterion_and_postprocessors(args)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
lwdetr = model.model.model
lwdetr.to(device)
criterion.to(device)

train_ds = COCODetectionDataset(str(IMAGES_DIR), TRAIN_ANN, RESOLUTION)
val_ds = COCODetectionDataset(str(IMAGES_DIR), VAL_ANN, RESOLUTION)

# High-throughput asynchronous DataLoaders with persistent workers & pinned memory
train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, collate_fn=collate_fn, drop_last=True,
    pin_memory=True, persistent_workers=(NUM_WORKERS > 0), prefetch_factor=2 if NUM_WORKERS > 0 else None
)
val_loader = DataLoader(
    val_ds, batch_size=max(BATCH_SIZE, 4), shuffle=False,
    num_workers=NUM_WORKERS, collate_fn=collate_fn,
    pin_memory=True, persistent_workers=(NUM_WORKERS > 0), prefetch_factor=2 if NUM_WORKERS > 0 else None
)

optimizer = torch.optim.AdamW(lwdetr.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

class BestValLossCheckpointSD:
    def __init__(self, checkpoint_dir: str, module: nn.Module, verbose: bool = True):
        self.ckpt_dir = Path(checkpoint_dir); self.ckpt_dir.mkdir(parents=True, exist_ok=True)
        self.module, self.verbose = module, verbose
        self.best_loss, self.best_epoch, self.history = float("inf"), -1, []
        self.log_path, self.best_path = self.ckpt_dir / "training_log.json", self.ckpt_dir / "best_model.pth"

    def step(self, epoch: int, train_loss: float, val_loss: float):
        self.history.append({"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss, "time": time.strftime("%Y-%m-%dT%H:%M:%S")})
        improved = val_loss < self.best_loss
        if improved:
            self.best_loss, self.best_epoch = val_loss, epoch
            torch.save(self.module.state_dict(), str(self.best_path))
            torch.save(self.module.state_dict(), str(self.ckpt_dir / f"epoch_{epoch:04d}_val{val_loss:.4f}.pth"))
        with open(self.log_path, "w") as f:
            json.dump({"best_epoch": self.best_epoch, "best_val_loss": self.best_loss, "history": self.history}, f, indent=2)
        marker = " <-- [BEST MODEL SAVED]" if improved else ""
        logger.info(f"Epoch {epoch:>4d}/{EPOCHS} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Best: {self.best_loss:.4f}{marker}")

    def load_best(self):
        self.module.load_state_dict(torch.load(str(self.best_path), map_location="cpu"))
        logger.info(f"Loaded best checkpoint (Epoch {self.best_epoch}, Val Loss: {self.best_loss:.4f})")

ckpt = BestValLossCheckpointSD(CHECKPOINT_DIR, lwdetr)
logger.info(f"DataLoaders: Train batches={len(train_loader)} (B={BATCH_SIZE}), Val batches={len(val_loader)} (B={max(BATCH_SIZE, 4)})")
print_vram_usage("Pre-Training")

In [ ]:
# CELL 17 - Optimized Training Step (AMP, Accumulation) & Evaluation
scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

def run_epoch(model, loader, criterion, optimizer, device, is_train: bool, epoch: int, num_epochs: int) -> float:
    model.train(is_train)
    phase = "train" if is_train else "val"
    total, count = 0.0, 0
    pbar = tqdm(loader, desc=f"Epoch {epoch:>4}/{num_epochs} [{phase}]", leave=False, unit="batch", dynamic_ncols=True)
    log_interval = min(max(len(loader) // 4, 1), 50)

    with torch.set_grad_enabled(is_train):
        for step, (images, targets) in enumerate(pbar):
            images = images.to(device, non_blocking=True)
            targets = [{k: v.to(device, non_blocking=True) for k, v in t.items()} for t in targets]

            with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
                outputs = model(images)
                loss_dict = criterion(outputs, targets)
                loss = sum(loss_dict[k] * criterion.weight_dict[k] for k in loss_dict if k in criterion.weight_dict)

            if is_train:
                loss_scaled = loss / GRAD_ACCUMULATION
                scaler.scale(loss_scaled).backward()
                if (step + 1) % GRAD_ACCUMULATION == 0 or (step + 1) == len(loader):
                    scaler.unscale_(optimizer)
                    nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.1)
                    scaler.step(optimizer)
                    scaler.update()
                    optimizer.zero_grad(set_to_none=True)

            total += loss.item()
            count += 1
            running_avg = total / count
            pbar.set_postfix({"loss": f"{running_avg:.4f}"}); pbar.update()

            if (step + 1) % log_interval == 0 or (step + 1) == len(loader):
                logger.info(f"Epoch {epoch:>4}/{num_epochs} [{phase}] Step {step+1:>4}/{len(loader)} | Loss: {loss.item():.4f} (Avg: {running_avg:.4f})")

    pbar.close()
    if torch.cuda.is_available() and not is_train:
        torch.cuda.empty_cache()  # Only clean cache once at end of validation
    epoch_avg = total / max(count, 1)
    logger.info(f"Epoch {epoch:>4}/{num_epochs} [{phase}] Complete | Avg Loss: {epoch_avg:.4f}")
    return epoch_avg

def box_cxcywh_to_xyxy(boxes: torch.Tensor) -> torch.Tensor:
    cx, cy, w, h = boxes.unbind(-1)
    return torch.stack([cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2], dim=-1)

def compute_map(model, loader, device, img_size: int, threshold: float = 0.3) -> dict:
    model.eval()
    metric = MeanAveragePrecision(iou_type="bbox", class_metrics=True)
    with torch.no_grad():
        for images, targets in loader:
            images = images.to(device, non_blocking=True)
            with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
                outputs = model(images)
            pred_logits, pred_boxes = outputs["pred_logits"], outputs["pred_boxes"]
            preds_list, tgts_list = [], []
            for i in range(len(images)):
                scores, lbs = pred_logits[i].sigmoid().max(-1)
                keep = scores > threshold
                preds_list.append({
                    "boxes": (box_cxcywh_to_xyxy(pred_boxes[i][keep]) * img_size).cpu().float(),
                    "scores": scores[keep].cpu().float(),
                    "labels": lbs[keep].cpu(),
                })
                tgts_list.append({
                    "boxes": (box_cxcywh_to_xyxy(targets[i]["boxes"]) * img_size).cpu().float(),
                    "labels": targets[i]["labels"].cpu(),
                })
            metric.update(preds_list, tgts_list)
    return metric.compute()

logger.info("Compiled optimized training and compute_map steps.")

In [ ]:
# CELL 18 - Fine-Tuning Execution Loop
logger.info("=" * 70)
logger.info(f"[CELL 18] Starting Multi-Class Training for {EPOCHS} Epochs (Batch: {BATCH_SIZE}, Effective: {BATCH_SIZE*GRAD_ACCUMULATION})...")
logger.info(f"Streaming logs actively to: {log_file_path}")
logger.info("=" * 70)

for epoch in range(1, EPOCHS + 1):
    tr_loss = run_epoch(lwdetr, train_loader, criterion, optimizer, device, is_train=True, epoch=epoch, num_epochs=EPOCHS)
    val_loss = run_epoch(lwdetr, val_loader, criterion, optimizer, device, is_train=False, epoch=epoch, num_epochs=EPOCHS)
    scheduler.step()
    ckpt.step(epoch, tr_loss, val_loss)

logger.info("=" * 70)
logger.info("Training finished successfully!")
print_vram_usage("Post-Training Peak")

In [ ]:
# CELL 19 - Load Best Checkpoint & Compute Validation mAP
logger.info("=" * 70)
logger.info("[CELL 19] Loading best checkpoint and evaluating validation mAP...")
logger.info("=" * 70)

ckpt.load_best()
val_map_results = compute_map(lwdetr, val_loader, device, img_size=RESOLUTION, threshold=0.3)

logger.info("=" * 50)
logger.info("Validation mAP Metrics:")
for k, v in val_map_results.items():
    if isinstance(v, torch.Tensor):
        if v.numel() == 1:
            logger.info(f"   - {k:20s}: {v.item():.4f}")
        else:
            logger.info(f"   - {k:20s}: {[round(x, 4) for x in v.tolist()]}")
    else:
        logger.info(f"   - {k:20s}: {v}")
logger.info("=" * 50)

In [ ]:
# CELL 20 - Export Best Model to model/ Directory
logger.info("=" * 70)
logger.info("[CELL 20] Exporting best model checkpoint...")
logger.info("=" * 70)

exported_path = os.path.join(FINAL_MODEL_DIR, f"best_model_{MODE_TAG}.pth")
if os.path.exists(ckpt.best_path):
    shutil.copy2(ckpt.best_path, exported_path)
    logger.info(f"   - Canonical best model saved to: {exported_path} ({os.path.getsize(exported_path) / 1e6:.1f} MB)")
else:
    logger.warning(f"   Checkpoint not found at {ckpt.best_path}")

logger.info("Export completed.")

In [ ]:
# CELL 21 - Test Set Inference & Visual Predictions
logger.info("=" * 70)
logger.info(f"[CELL 21] Running inference on test split (Confidence={CONFIDENCE}, NMS={NMS_THRESHOLD})...")
logger.info("=" * 70)

import torch.multiprocessing as mp
try:
    mp.set_sharing_strategy('file_system')
except Exception:
    pass
try:
    import resource
    rlimit = resource.getrlimit(resource.RLIMIT_NOFILE)
    resource.setrlimit(resource.RLIMIT_NOFILE, (max(rlimit[0], 4096), max(rlimit[1], 4096)))
except Exception:
    pass

test_ds = COCODetectionDataset(str(IMAGES_DIR), TEST_ANN, RESOLUTION)
test_loader = DataLoader(test_ds, batch_size=1, shuffle=False, num_workers=0, collate_fn=collate_fn)

test_map_results = compute_map(lwdetr, test_loader, device, img_size=RESOLUTION, threshold=CONFIDENCE)
logger.info("Test Set mAP Results:")
for k, v in test_map_results.items():
    if isinstance(v, torch.Tensor):
        if v.numel() == 1:
            logger.info(f"   - {k:20s}: {v.item():.4f}")
        else:
            logger.info(f"   - {k:20s}: {[round(x, 4) for x in v.tolist()]}")

# Visual sample test predictions using supervision
color_palette = sv.ColorPalette.from_hex([
    "#e6194B", "#3cb44b", "#ffe119", "#4363d8", "#f58231",
    "#911eb4", "#42d4f4", "#f032e6", "#bfef45", "#fabed4", "#469990"
])
box_annotator = sv.BoxAnnotator(color=color_palette, thickness=2)
label_annotator = sv.LabelAnnotator(color=color_palette, text_scale=0.5, text_thickness=1)

lwdetr.eval()
preview_count = 0

with torch.no_grad():
    for images, targets in test_loader:
        if preview_count >= 3:
            break
        images = images.to(device)
        with torch.amp.autocast(device_type="cuda", enabled=torch.cuda.is_available()):
            outputs = lwdetr(images)
        pred_logits = outputs["pred_logits"][0]
        pred_boxes = outputs["pred_boxes"][0]

        scores, class_ids = pred_logits.sigmoid().max(-1)
        keep = scores > CONFIDENCE
        if keep.sum() == 0:
            continue

        boxes_xyxy = (box_cxcywh_to_xyxy(pred_boxes[keep]) * RESOLUTION).cpu().numpy()
        scores_np = scores[keep].cpu().numpy()
        class_ids_np = class_ids[keep].cpu().numpy()

        img_np = (images[0].permute(1, 2, 0).cpu().numpy() * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406]))
        img_np = np.clip(img_np * 255, 0, 255).astype(np.uint8)

        detections = sv.Detections(xyxy=boxes_xyxy, confidence=scores_np, class_id=class_ids_np)
        labels = [f"{class_names[cid] if cid < len(class_names) else str(cid)} {c:.2f}" for cid, c in zip(class_ids_np, scores_np)]
        annotated = box_annotator.annotate(scene=img_np.copy(), detections=detections)
        annotated = label_annotator.annotate(scene=annotated, detections=detections, labels=labels)

        plt.figure(figsize=(10, 6))
        plt.imshow(annotated)
        plt.title(f"Test Detection Preview ({len(boxes_xyxy)} boxes detected across {NUM_CLASSES} classes)")
        plt.axis("off")
        plt.show()
        preview_count += 1

logger.info(f"Previewed {preview_count} test detection results.")

In [ ]:
# CELL 22 - Save Test Predictions JSON
logger.info("=" * 70)
logger.info("[CELL 22] Saving test predictions...")
logger.info("=" * 70)

pred_out_file = os.path.join(INFERENCE_OUTPUT_DIR, f"test_predictions_{MODE_TAG}.json")
lwdetr.eval()
all_test_predictions = {}

with torch.no_grad():
    for images, targets in test_loader:
        img_id = targets[0]["image_id"].item()
        orig_h, orig_w = targets[0]["orig_size"].tolist()
        images = images.to(device)
        with torch.amp.autocast(device_type="cuda", enabled=torch.cuda.is_available()):
            outputs = lwdetr(images)
        pred_logits = outputs["pred_logits"][0]
        pred_boxes = outputs["pred_boxes"][0]

        scores, class_ids = pred_logits.sigmoid().max(-1)
        keep = scores > CONFIDENCE

        boxes_xyxy = (box_cxcywh_to_xyxy(pred_boxes[keep]) * np.array([orig_w, orig_h, orig_w, orig_h])).cpu().tolist()
        scores_list = scores[keep].cpu().tolist()
        class_ids_list = class_ids[keep].cpu().tolist()

        all_test_predictions[str(img_id)] = {
            "boxes_xyxy": boxes_xyxy,
            "confidence": scores_list,
            "class_id": class_ids_list,
            "orig_size": [orig_h, orig_w]
        }

with open(pred_out_file, "w", encoding="utf-8") as f:
    json.dump({
        "pipeline": PIPELINE_NAME,
        "mode": MODE_TAG,
        "num_classes": NUM_CLASSES,
        "class_names": class_names,
        "confidence_threshold": CONFIDENCE,
        "predictions": all_test_predictions
    }, f, indent=2)

logger.info(f"Saved test predictions to: {pred_out_file}")

In [ ]:
# CELL 23 - Final Execution Summary
logger.info("=" * 70)
logger.info(f"[CELL 23] Execution Summary: {PIPELINE_NAME}")
logger.info("=" * 70)
logger.info(f"   - Pipeline Mode:        {MODE_TAG.upper()}")
logger.info(f"   - Number of Classes:    {NUM_CLASSES} ({class_names})")
logger.info(f"   - Architecture:         RF-DETR-{MODEL_SIZE.capitalize()} (resolution={RESOLUTION})")
logger.info(f"   - Effective Batch Size: {BATCH_SIZE * GRAD_ACCUMULATION} (Batch={BATCH_SIZE}, Accum={GRAD_ACCUMULATION})")
logger.info(f"   - Learning Rate:        {LEARNING_RATE} (Weight Decay: {WEIGHT_DECAY})")
logger.info(f"   - Dataset Folder:       {DATASET_DIR}")
logger.info(f"   - Images Folder:        {IMAGES_DIR} (Zero-copy shared repository)")
logger.info(f"   - Checkpoint Folder:    {CHECKPOINT_DIR}")
logger.info(f"   - Exported Model:       {exported_path}")
logger.info(f"   - Test Predictions:     {pred_out_file}")
logger.info(f"   - Pipeline Log File:    {log_file_path}")
logger.info("=" * 70)
logger.info("Multi-Class Pipeline execution completed successfully!")